## Load modules

In [1]:
import sys
sys.path.append('/home/bowen/Workspace/Github/CosyVoice/third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import CosyVoice, CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio


/home/bowen/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Model and prmopt voice

In [2]:
cosyvoice = CosyVoice2('/home/bowen/Workspace/Github/CosyVoice/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False, use_flow_cache=False)

# prompt_speech_16k = load_wav('/home/bowen/Workspace/Github/CosyVoice/asset/zero_shot_prompt.wav', 16000)


/home/bowen/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-05-09 02:20:34,172 INFO input frame rate=25
/home/bowen/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
2025-05-09 02:20:3

text.cc: festival_Text_init
open voice lang map failed


### Given Demo

In [ ]:
for i, j in enumerate(cosyvoice.inference_instruct2('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '用四川话说这句话', prompt_speech_16k, stream=False)):
    torchaudio.save('instruct_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_zero_shot(text_generator(), '希望你以后能够做的比我还好呦。', prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [4]:
out = cosyvoice.inference_instruct2('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '非常快速', prompt_speech_16k, stream=False)


## Please try this demo

In [10]:
# Load model and prmopt audio
# cosyvoice = CosyVoice2('/home/bowen/Workspace/Github/CosyVoice/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False, use_flow_cache=False)
prompt_speech_16k = load_wav('asset/US_M_prompt.wav', 16000)

text = '就我觉得  three  years  then  some  more  它是一个  professional  job  then  我就觉得  i  mean  like  why  spend  four  years  doing  a  general  arts.'
# text = "jiu wo jue de  three  years  then  some  more  ta shi yi ge  professional  job  then  wo jiu jue de  i  mean  like  why  spend  four  years  doing  a  general  arts"
style = '用新加坡人的口音说'

# Get the generator and extract the first result
audio_generator = cosyvoice.inference_instruct2(
    tts_text=text, 
    instruct_text=style,
    prompt_speech_16k=prompt_speech_16k,
    stream=False
)

# Get the first (and likely only) item from the generator
audio_output = next(audio_generator)

# Save the audio file
output_path = 'output/US_py_M_sg_accent.wav'
torchaudio.save(
    output_path, 
    audio_output['tts_speech'], 
    cosyvoice.sample_rate
)

print(f"Audio saved to {output_path}")

  0%|          | 0/1 [00:00<?, ?it/s]2025-05-08 16:40:39,916 INFO synthesis text jiu wo jue de three years then some more ta shi yi ge professional job then wo jiu jue de i mean like why spend four years doing a general arts.
2025-05-08 16:40:49,720 INFO yield speech len 11.76, rtf 0.833748635791597


Audio saved to output/US_py_M_sg_accent.wav


### Try to remove prompt speech

In [ ]:
# check available speakers
try:
    available_speakers = cosyvoice.list_available_spks()
    print(f"Available speakers: {available_speakers}")
    
    # Find Chinese male speakers if available
    chinese_male_speakers = [spk for spk in available_speakers if '中文' in spk or '男' in spk]
    print(f"Chinese male speakers: {chinese_male_speakers}")
    
    # Select a speaker ID
    speaker_id = chinese_male_speakers[0] if chinese_male_speakers else available_speakers[0]
    print(f"Using speaker ID: {speaker_id}")
    
    # Try the inference with the selected speaker
    audio_generator = cosyvoice.inference_zero_shot(
        tts_text=text,
        prompt_text=style,
        prompt_speech_16k=None,
        zero_shot_spk_id=speaker_id,
        stream=False
    )
    
    # Get the output
    audio_output = next(audio_generator)
    
    # Save the audio file
    output_path = 'generated_audio_fixed.wav'
    torchaudio.save(
        output_path, 
        audio_output['tts_speech'], 
        cosyvoice.sample_rate
    )
    
    print(f"Audio saved to {output_path}")
except Exception as e:
    print(f"Detailed error in modified approach: {str(e)}")

Available speakers: ['中文女', '中文男', '日语男', '粤语女', '英文女', '英文男', '韩语女', 'my_zero_shot_spk']
Chinese male speakers: ['中文女', '中文男', '日语男', '英文男']
Using speaker ID: 中文女


  0%|          | 0/1 [00:05<?, ?it/s]


Detailed error in modified approach: expand(torch.cuda.FloatTensor{[0, 80]}, size=[80]): the number of sizes provided (1) must be greater or equal to the number of dimensions in the tensor (2)

Trying alternative approach...
Testing with ID: zh_001


  0%|          | 0/1 [00:00<?, ?it/s]


Failed with ID zh_001: 'zh_001'
Testing with ID: zh_male


...

Failed with ID chinese_male: 'chinese_male'
Testing with ID: male_zh


  0%|          | 0/1 [00:00<?, ?it/s]


Failed with ID male_zh: 'male_zh'
Testing with ID: default


  0%|          | 0/1 [00:00<?, ?it/s]

Failed with ID default: 'default'


quickly check cosyvoice2
 
transcript: 就  我  觉得  three  years  then  some  more  它是  一个  professional  job  then  我  就  觉得  i  mean  like  why  spend  four  years  doing  a  general  arts
style: chinese accent
 
transcript: 就  我  觉得  three  years  then  some  more  它是  一个  professional  job  then  我  就  觉得  i  mean  like  why  spend  four  years  doing  a  general  arts
style: singapore accent
 
transcript: 就  我  觉得  three  years  then  some  more  它是  一个  professional  job  then  我  就  觉得  i  mean  like  why  spend  four  years  doing  a  general  arts
style: NONE
 

In [25]:
# Load model and prmopt audio
# cosyvoice = CosyVoice2('/home/bowen/Workspace/Github/CosyVoice/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False, use_flow_cache=False)
# prompt_speech_16k = load_wav('/home/bowen/Workspace/Github/CosyVoice/asset/zero_shot_prompt.wav', 16000)

text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'
style = '非常快速'

# Get the generator and extract the first result
audio_generator = cosyvoice.inference_instruct2(
    tts_text=text, 
    instruct_text=style,
    prompt_speech_16k=None,
    zero_shot_spk_id='中文男',
    stream=False
)

# Get the first (and likely only) item from the generator
audio_output = next(audio_generator)

# Save the audio file
output_path = 'generated_audio_male.wav'
torchaudio.save(
    output_path, 
    audio_output['tts_speech'], 
    cosyvoice.sample_rate
)

print(f"Audio saved to {output_path}")

  0%|          | 0/1 [00:00<?, ?it/s]


KeyError: 'llm_prompt_speech_token'

## Web Demo

In [3]:
import sys
import gradio as gr
sys.path.append('third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio
import torch

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False)

def generate_audio(audio_path, tts_text, instruct_text):
    if not audio_path or not tts_text or not instruct_text:
        return None
        
    prompt_speech = load_wav(audio_path, 16000)
    
    # 生成音频
    results = []
    for i, j in enumerate(cosyvoice.inference_instruct2(
        tts_text, 
        instruct_text,
        prompt_speech,
        stream=False
    )):
        output_path = f"output/test/output_{i}.wav"
        torchaudio.save(output_path, j['tts_speech'], cosyvoice.sample_rate)
        results.append(output_path)
    
    if not results:
        return None
        
    # 拼接所有音频
    waveforms = []
    for path in results:
        waveform, sr = torchaudio.load(path)
        waveforms.append(waveform)
    
    concatenated = torch.cat(waveforms, dim=1)
    output_path = "output_combined.wav"
    torchaudio.save(output_path, concatenated, cosyvoice.sample_rate)
    
    return output_path

with gr.Blocks(title="CosyVoice TTS") as app:
    gr.Markdown("## CosyVoice 语音合成系统")
    
    with gr.Row():
        with gr.Column():
            ref_audio = gr.Audio(label="参考音频", type="filepath")
            tts_text = gr.Textbox(label="合成文本", placeholder="输入要合成的文本...")
            instruct_text = gr.Textbox(label="风格指令", placeholder="输入语音风格指令...")
            generate_btn = gr.Button("生成语音", variant="primary")
        
        with gr.Column():
            audio_output = gr.Audio(label="生成结果", interactive=False)

    generate_btn.click(
        fn=generate_audio,
        inputs=[ref_audio, tts_text, instruct_text],
        outputs=audio_output
    )

if __name__ == "__main__":
    app.launch(server_name="0.0.0.0", server_port=17860, share=True)

2025-05-09 02:27:06,114 DEBUG connect_tcp.started host='api.gradio.app' port=443 local_address=None timeout=3 socket_options=None
2025-05-09 02:27:06,218 DEBUG Importing BlpImagePlugin
2025-05-09 02:27:06,220 DEBUG Importing BmpImagePlugin
2025-05-09 02:27:06,221 DEBUG Importing BufrStubImagePlugin
2025-05-09 02:27:06,222 DEBUG Importing CurImagePlugin
2025-05-09 02:27:06,223 DEBUG Importing DcxImagePlugin
2025-05-09 02:27:06,224 DEBUG Importing DdsImagePlugin
2025-05-09 02:27:06,228 DEBUG Importing EpsImagePlugin
2025-05-09 02:27:06,230 DEBUG Importing FitsImagePlugin
2025-05-09 02:27:06,231 DEBUG Importing FliImagePlugin
2025-05-09 02:27:06,232 DEBUG Importing FpxImagePlugin
2025-05-09 02:27:06,233 DEBUG Image: failed to import FpxImagePlugin: No module named 'olefile'
2025-05-09 02:27:06,234 DEBUG Importing FtexImagePlugin
2025-05-09 02:27:06,235 DEBUG Importing GbrImagePlugin
2025-05-09 02:27:06,236 DEBUG Importing GifImagePlugin
2025-05-09 02:27:06,238 DEBUG Importing GribStubImag

open voice lang map failed


2025-05-09 02:27:16,049 DEBUG Starting new HTTPS connection (1): huggingface.co:443
2025-05-09 02:27:16,061 DEBUG connect_tcp.started host='api.gradio.app' port=443 local_address=None timeout=3 socket_options=None
2025-05-09 02:27:16,380 DEBUG https://huggingface.co:443 "HEAD /api/telemetry/gradio/initiated HTTP/11" 200 0
2025-05-09 02:27:16,380 DEBUG connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f78c0384b80>
2025-05-09 02:27:16,384 DEBUG start_tls.started ssl_context=<ssl.SSLContext object at 0x7f78ae3f7a40> server_hostname='api.gradio.app' timeout=3
2025-05-09 02:27:16,608 DEBUG Using selector: EpollSelector
2025-05-09 02:27:16,638 DEBUG connect_tcp.started host='localhost' port=17860 local_address=None timeout=None socket_options=None
2025-05-09 02:27:16,640 DEBUG connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f78ad60fc10>
2025-05-09 02:27:16,642 DEBUG send_request_headers.started request=<Request [b'GET']>
202

* Running on local URL:  http://0.0.0.0:17860


2025-05-09 02:27:16,951 DEBUG receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Fri, 09 May 2025 02:27:16 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'21'), (b'Connection', b'keep-alive'), (b'Server', b'nginx/1.18.0'), (b'Access-Control-Allow-Origin', b'*')])
2025-05-09 02:27:16,952 DEBUG connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f78c8494fa0>
2025-05-09 02:27:16,953 INFO HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2025-05-09 02:27:16,955 DEBUG start_tls.started ssl_context=<ssl.SSLContext object at 0x7f78ae3f01c0> server_hostname='api.gradio.app' timeout=30
2025-05-09 02:27:16,956 DEBUG receive_response_body.started request=<Request [b'GET']>
2025-05-09 02:27:16,960 DEBUG receive_response_body.complete
2025-05-09 02:27:16,962 DEBUG response_closed.started
2025-05-09 02:27:16,963 DEBUG response_closed.complete
2025-05-09 02:27:16,964 DEBUG close.started
2025-05

* Running on public URL: https://a9e07fcc95e7aab0ff.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2025-05-09 02:27:18,584 DEBUG connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f78c8496110>
2025-05-09 02:27:18,586 DEBUG start_tls.started ssl_context=<ssl.SSLContext object at 0x7f78ae3f0840> server_hostname='a9e07fcc95e7aab0ff.gradio.live' timeout=3
2025-05-09 02:27:18,963 DEBUG start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f78ad5b0cd0>
2025-05-09 02:27:18,965 DEBUG send_request_headers.started request=<Request [b'HEAD']>
2025-05-09 02:27:18,966 DEBUG send_request_headers.complete
2025-05-09 02:27:18,968 DEBUG send_request_body.started request=<Request [b'HEAD']>
2025-05-09 02:27:18,970 DEBUG send_request_body.complete
2025-05-09 02:27:18,971 DEBUG receive_response_headers.started request=<Request [b'HEAD']>
2025-05-09 02:27:19,353 DEBUG receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Fri, 09 May 2025 02:27:19 GMT'), (b'Content-Type', b'text/html; charset=utf-8'), (b'Content-Le

2025-05-09 02:27:19,378 DEBUG Starting new HTTPS connection (1): huggingface.co:443


2025-05-09 02:27:19,622 DEBUG https://huggingface.co:443 "HEAD /api/telemetry/gradio/launched HTTP/11" 200 0
2025-05-09 02:27:48,615 DEBUG Calling on_part_begin with no data
2025-05-09 02:27:48,617 DEBUG Calling on_header_field with data[42:61]
2025-05-09 02:27:48,618 DEBUG Calling on_header_value with data[63:108]
2025-05-09 02:27:48,620 DEBUG Calling on_header_end with no data
2025-05-09 02:27:48,622 DEBUG Calling on_header_field with data[110:122]
2025-05-09 02:27:48,625 DEBUG Calling on_header_value with data[124:148]
2025-05-09 02:27:48,626 DEBUG Calling on_header_end with no data
2025-05-09 02:27:48,629 DEBUG Calling on_headers_finished with no data
2025-05-09 02:27:48,632 DEBUG Calling on_part_data with data[152:16375]
2025-05-09 02:27:48,803 DEBUG Calling on_part_data with data[0:16375]
2025-05-09 02:27:48,882 DEBUG Calling on_part_data with data[0:16375]
2025-05-09 02:27:48,992 DEBUG Calling on_part_data with data[0:16411]
2025-05-09 02:27:48,994 DEBUG Calling on_part_data wit